# 03 · Retrieval — Keyword (BM25) vs Semantic
Two ways to find relevant documents: BM25 (exact keywords) and semantic (meaning via vectors). We load the practice corpus and try both.

In [ ]:
# A REAL but simple vectorizer: bag-of-words (term frequency). Offline, deterministic,
# and good enough that similar documents (sharing words) get similar vectors — so
# cosine similarity and retrieval metrics are genuinely meaningful and hand-checkable.
#
# NOTE: production RAG uses NEURAL embeddings (e.g. Jina, OpenAI, sentence-transformers)
# that also capture SYNONYMS ("car" ~ "automobile") with no shared words. The MATH below
# (cosine, retrieval, metrics) is identical; only the vectors get smarter. Where a cell
# says "swap in a real embedder", that's the one line that changes.
import numpy as np, re
def tokenize(text): return re.findall(r"[a-z0-9]+", text.lower())
def build_vocab(texts):
    vocab={}
    for t in texts:
        for w in tokenize(t):
            if w not in vocab: vocab[w]=len(vocab)
    return vocab
def vectorize(text, vocab):
    v=np.zeros(len(vocab))
    for w in tokenize(text):
        if w in vocab: v[vocab[w]]+=1.0
    return v
def cosine(a,b):
    na,nb=np.linalg.norm(a),np.linalg.norm(b)
    return float(a@b/(na*nb)) if na and nb else 0.0

## 1. Load the corpus

In [ ]:
import pandas as pd
docs = pd.read_csv("../dataset/corpus.csv")
print(len(docs), "documents")
docs.head()

## 2. Semantic retrieval (embeddings + cosine)

In [ ]:
vocab = build_vocab(docs["text"].tolist())
doc_vecs = [vectorize(t, vocab) for t in docs["text"]]

def semantic_search(query, k=3):
    qv = vectorize(query, vocab)
    scored = sorted(
        [(docs["id"][i], cosine(qv, doc_vecs[i]), docs["text"][i]) for i in range(len(docs))],
        key=lambda x: -x[1])
    return scored[:k]

for did, score, text in semantic_search("how do I get a refund"):
    print(f"  {did}  {score:.3f}  {text[:55]}")

## 3. Keyword retrieval (BM25)
BM25 scores by term overlap, rewarding rare shared words and discounting very common ones. We use the `rank_bm25` library if present, else a simple fallback.

In [ ]:
try:
    from rank_bm25 import BM25Okapi
    corpus_tokens = [tokenize(t) for t in docs["text"]]
    bm25 = BM25Okapi(corpus_tokens)
    def bm25_search(query, k=3):
        scores = bm25.get_scores(tokenize(query))
        order = sorted(range(len(docs)), key=lambda i: -scores[i])[:k]
        return [(docs["id"][i], round(float(scores[i]),3), docs["text"][i]) for i in order]
    engine = "rank_bm25"
except ImportError:
    # simple fallback: overlap count (teaches the idea without the library)
    def bm25_search(query, k=3):
        qt=set(tokenize(query))
        scored=[(docs["id"][i], len(qt & set(tokenize(docs["text"][i]))), docs["text"][i]) for i in range(len(docs))]
        return sorted(scored, key=lambda x:-x[1])[:k]
    engine = "fallback-overlap (pip install rank_bm25 for real BM25)"

print("engine:", engine)
for did, score, text in bm25_search("how do I get a refund"):
    print(f"  {did}  {score}  {text[:55]}")

**Observe:** both find the refund docs here because the query shares words with them. The difference shows on *paraphrases*: ask 'I want my money back' and BM25 struggles (no shared words with 'refund'), while a real semantic embedder still finds the refund docs. That's the core trade-off:

```
  BM25 (keyword)        Semantic (embeddings)
  ---------------       ---------------------
  exact words match     meaning matches
  great for codes,      great for paraphrases,
  names, IDs            synonyms, questions
  no model needed       needs an embedding model
```
**Hybrid search** (a later concept) fuses both to get the best of each.